In [1]:
import pandas as pd
import numpy as np
import time

df_crimes = pd.read_csv(
    "./output/maceio/maceio_with_grid_index.csv",
    delimiter=";",
    parse_dates=["DATA_HORA"]
)
crimes_list=[
    "street_robbery",
    "vehicle_robbery",
    "public_transport_robbery",
    "commercial_robbery",
    "residential_robbery",
]

# If granularity = 24 (1 step = 1 day)
# 7 days lookback: n_frames = 7 + 1 = 8
# 14 days lookback: n_frames = 14 + 1 = 15
#
# If granularity = 12 (1 step = 12 hours; 2 steps/day)
# 7 days lookback: lookback_steps = 7×2 = 14 → n_frames = 15
# 14 days lookback: 14×2 = 28 → n_frames = 29
#
# If granularity = 8 (1 step = 8 hours; 3 steps/day)
# 7 days lookback: 7×3 = 21 → n_frames = 22
# 14 days lookback: 14×3 = 42 → n_frames = 43
granularity = 24
n_frames = 8
grid_size = 73
grid_size_sub = 16
num_sub = 5

In [2]:

start_year = 2012
end_year = 2022

import calendar

years = list(range(start_year, end_year + 1))

def get_granularity_year_range_index(granularity_hours):
    if granularity_hours == 24:
        return 1
    elif granularity_hours == 12:
        return 2
    elif granularity_hours == 8:
        return 3
    elif granularity_hours == 6:
        return 4
    else:   
        raise ValueError("Granularity not supported")

def periods_in_year(year, granularity_hours):
    days = 366 if calendar.isleap(year) else 365
    return (days * get_granularity_year_range_index(granularity_hours))

idx_list = []
for y in years:
    n = periods_in_year(y, granularity)
    idx_list += [f'{i}_{y}' for i in range(n)]
df_cri_list = []
for crime_type in crimes_list:
    df_cri_temp = pd.read_csv(f"./output/maceio/maceio_{crime_type}_all_grid.csv",index_col=0)
    df_cri_temp.reset_index(inplace=True,drop=True)
    df_cri_temp = df_cri_temp.groupby(df_cri_temp.index//granularity).sum()
    df_cri_temp.index = idx_list
    df_cri_list.append(df_cri_temp)

display(df_cri_list)
# so why separate it before?
# add the counts of each crime type
df_cri = df_cri_list[0]
for i in range(1,len(crimes_list)):
    df_cri += df_cri_list[i]

# NORMALLIZE TO CRIME/NO CRIME
df_cri[df_cri >= 1] = 1 
print("Crime dataset shape: ", df_cri.shape)


[            0    1    2    3    4    5    6    7    8    9  ...  5319  5320  \
 0_2012    0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  ...   0.0   0.0   
 1_2012    0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  ...   0.0   0.0   
 2_2012    0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  ...   0.0   0.0   
 3_2012    0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  ...   0.0   0.0   
 4_2012    0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  ...   0.0   0.0   
 ...       ...  ...  ...  ...  ...  ...  ...  ...  ...  ...  ...   ...   ...   
 360_2022  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  ...   0.0   0.0   
 361_2022  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  ...   0.0   0.0   
 362_2022  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  ...   0.0   0.0   
 363_2022  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  ...   0.0   0.0   
 364_2022  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  ...   0.0   0.0   
 
           5321  5322  5323  5324  532

Crime dataset shape:  (4018, 5329)


In [3]:
# normalize the data
def NormalizeData(data):
  n = (data - np.min(data)) / (np.max(data) - np.min(data))
  if np.isnan(n).all(): # because if we have all 0's then it turn into array of nan
    n = np.nan_to_num(n)
  return n

def NormalizeDataLocal(data,mini,maxi): # can use this if code is too slow
  n = (data - mini) / (maxi - mini)
  if np.isnan(n).all(): # because if we have all 0's then it turn into array of nan
    n = np.nan_to_num(n)
  return n

## Put data into the correct shape
def extract_one_variable(df,variable,grid_size,normalize=False,mini=None,maxi=None):
  df_var_list = df.loc[variable].values.tolist()
  df_var_list.reverse()
  m = np.array(df_var_list).reshape(grid_size,grid_size,1).T # populate in correct order and add n_samples dimension
  if normalize:
    m = NormalizeDataLocal(m,mini,maxi)
  return np.expand_dims(m, axis=-1) # we add the channel dimension


## Make full matrix for each hour (CRIME ONLY)
def make_full_matrix_per_hour(hour_var, df_cri, grid_size):
  list_ch = []
  crime_m = extract_one_variable(
    df=df_cri, 
    variable=hour_var,
    grid_size=grid_size,
    normalize=False
  )
  list_ch.append(crime_m)

  # axis=3 means we merge in the fourth dimension
  final = np.concatenate(list_ch,axis=3)
  return final

start_time = time.time()
# we use a sliding window here (so n_samples==number of sequences)
n_samples = df_cri.shape[0] - n_frames+1
print(f"We should get {n_samples} samples")

# make the full dataset (uses year from hour_var)
full_dataset_l = []
for var in df_cri.index.tolist(): # var is the index names I set in the function load_preprocessed_data
    full_dataset_l.append(
      make_full_matrix_per_hour(
        hour_var=var,
        df_cri=df_cri,
        grid_size=grid_size
      )
    )
full_dataset = np.concatenate(full_dataset_l,axis=0)
print("Shape full dataset: ", full_dataset.shape)
display(full_dataset)
end_time = time.time()
print(f"Time taken to generate full dataset: {end_time - start_time} seconds")

# Create a view of the dataset with a sliding window using NumPy's stride_tricks
start_time = time.time()
shape = (n_samples, n_frames, *full_dataset.shape[1:])
strides = (full_dataset.strides[0], *full_dataset.strides)
dataset = np.lib.stride_tricks.as_strided(full_dataset, shape=shape, strides=strides, writeable=False)
print("Shape dataset after sliding window: ", dataset.shape)
end_time = time.time()
print(f"Time taken to generate the sequences: {end_time - start_time} seconds")

# Chronological splitting with buffer of n_frames-1 between train and validation
start_time = time.time()
num_samples = dataset.shape[0]
train_size = int(0.9 * num_samples)

buffer = n_frames - 1  # we have to skip n_frames-1 samples to make sure there's no overlap between train and test
train_dataset = dataset[:train_size - buffer].copy()
val_dataset = dataset[train_size:].copy()
end_time = time.time()
print("shape train_dataset:", train_dataset.shape) # to make sure we apply the mask properly
print("shape val_dataset:", val_dataset.shape)

print(f"Number of samples={num_samples}, train_size={train_size}, buffer={buffer}, shape train_dataset={train_dataset.shape}, shape val_dataset={val_dataset.shape}")

print(f"Time taken to split train and validation (chronological with buffer): {end_time - start_time} seconds")

# extract min and max for each feature
start_time = time.time()
min_vals = np.nanmin(train_dataset, axis=(0,1,2,3), keepdims=True)
max_vals = np.nanmax(train_dataset, axis=(0,1,2,3), keepdims=True)
end_time = time.time()
print(f"Time taken to calculate min and max: {end_time - start_time} seconds")

# # load masks for NaN values
mask = np.load(f"./output/maceio/maceio_mask_final.npy")

start_time = time.time()
mask_broadcasted_train = np.broadcast_to(mask[None, None, :, :, None], train_dataset.shape)
train_dataset[mask_broadcasted_train == 0] = 0
mask_broadcasted_val = np.broadcast_to(mask[None, None, :, :, None], val_dataset.shape)
val_dataset[mask_broadcasted_val == 0] = 0
end_time = time.time()
print(f"Time taken to replace nan with 0: {end_time - start_time} seconds")

# NORMALIZATION IS DONE HERE USING MIN/MAX FROM TRAIN SET ALWAYS
# normalize the train_dataset in-place
start_time = time.time()
np.subtract(train_dataset, min_vals, out=train_dataset)
np.divide(train_dataset, max_vals - min_vals, out=train_dataset)
# normalize the val_dataset in-place
np.subtract(val_dataset, min_vals, out=val_dataset)
np.divide(val_dataset, max_vals - min_vals, out=val_dataset)
end_time = time.time()
print(f"Time taken to perform the min-max normalization: {end_time - start_time} seconds")

# Restore NaN values in both datasets (so NaN can be tracked in the algorithm to apply mask before metric)
train_dataset[mask_broadcasted_train == 0] = np.nan
val_dataset[mask_broadcasted_val == 0] = np.nan

def create_shifted_frames(data):
    x = data[:,:-1,:,:,:] #we remove the last frame
    y = data[:,-1,:,:,0] # we take only the crime channel from the last frame
    y = np.expand_dims(y, axis=-1)
    return x, y

start_time = time.time()
# Apply the processing function to the datasets.
x_train, y_train = create_shifted_frames(train_dataset)
x_val, y_val = create_shifted_frames(val_dataset)
end_time = time.time()
print(f"Time taken to make shifted frames: {end_time - start_time} seconds")

# Inspect the dataset.
print("Training Dataset Shapes: " + str(x_train.shape) + ", " + str(y_train.shape))
print("Validation Dataset Shapes: " + str(x_val.shape) + ", " + str(y_val.shape))

# return (x_train,y_train,x_val,y_val)

We should get 4011 samples
Shape full dataset:  (4018, 73, 73, 1)


array([[[[0.],
         [0.],
         [0.],
         ...,
         [0.],
         [0.],
         [0.]],

        [[0.],
         [0.],
         [0.],
         ...,
         [0.],
         [0.],
         [0.]],

        [[0.],
         [0.],
         [0.],
         ...,
         [0.],
         [0.],
         [0.]],

        ...,

        [[0.],
         [0.],
         [0.],
         ...,
         [0.],
         [0.],
         [0.]],

        [[0.],
         [0.],
         [0.],
         ...,
         [0.],
         [0.],
         [0.]],

        [[0.],
         [0.],
         [0.],
         ...,
         [0.],
         [0.],
         [0.]]],


       [[[0.],
         [0.],
         [0.],
         ...,
         [0.],
         [0.],
         [0.]],

        [[0.],
         [0.],
         [0.],
         ...,
         [0.],
         [0.],
         [0.]],

        [[0.],
         [0.],
         [0.],
         ...,
         [0.],
         [0.],
         [0.]],

        ...,

        [[0.],
 

Time taken to generate full dataset: 13.655189990997314 seconds
Shape dataset after sliding window:  (4011, 8, 73, 73, 1)
Time taken to generate the sequences: 0.0001652240753173828 seconds
shape train_dataset: (3602, 8, 73, 73, 1)
shape val_dataset: (402, 8, 73, 73, 1)
Number of samples=4011, train_size=3609, buffer=7, shape train_dataset=(3602, 8, 73, 73, 1), shape val_dataset=(402, 8, 73, 73, 1)
Time taken to split train and validation (chronological with buffer): 2.0060389041900635 seconds
Time taken to calculate min and max: 0.11729907989501953 seconds
Time taken to replace nan with 0: 0.4076194763183594 seconds
Time taken to perform the min-max normalization: 0.18657779693603516 seconds
Time taken to make shifted frames: 7.510185241699219e-05 seconds
Training Dataset Shapes: (3602, 7, 73, 73, 1), (3602, 73, 73, 1)
Validation Dataset Shapes: (402, 7, 73, 73, 1), (402, 73, 73, 1)


In [4]:
def generate_random_submatrices_fast(grid_size,grid_size_sub,num_sub,x,y):

  num_samples = x.shape[0]
  # we add a frame dimension
  y = np.expand_dims(y, axis=1)

  x_sub_list = []
  y_sub_list = []
  index_sub_list = []

  for sample in range(num_samples):
    count = 0
    it = 0
    while count < num_sub:

      # generate random starting indices
      i = np.random.randint(0, grid_size - grid_size_sub)
      j = np.random.randint(0, grid_size - grid_size_sub)

      # Extract 16x16 sections until we reach num_sub
      y_sub = y[sample,:,i:i+grid_size_sub, j:j+grid_size_sub,:]

      sum_num = np.nansum(np.array(y_sub)[0,:,:,0].flatten().tolist())

      min_num_crimes = 1
      if sum_num>=min_num_crimes:
        x_sub_list.append(x[sample,:,i:i+grid_size_sub, j:j+grid_size_sub,:].tolist())
        y_sub_list.append(y_sub.tolist())
        index_sub_list.append((i,j)) # save the indexes
        count += 1
      elif sum_num <=min_num_crimes:
        it +=1
        if it >= 50: # set a maximum amount of iterations before we give up on that sample
          print(f"Sample {sample}: couldn't find submatrix! Count={count}")
          break
      elif np.isnan(sum_num): # otherwise code got stuck here for all nan observations
        print(f"Sample {sample}: is all nan!")
        break

  print("Submatrices created!")
  return (np.array(x_sub_list),np.array(y_sub_list)[:,0,:,:,:],index_sub_list)

# generate random submatrices for each set
print("\n3. Generate random submatrices...")
start_time = time.time()
x_t_sub,y_t_sub,index_sub_t = generate_random_submatrices_fast(grid_size,grid_size_sub,num_sub,x_train,y_train)
x_v_sub,y_v_sub,index_sub_v = generate_random_submatrices_fast(grid_size,grid_size_sub,num_sub,x_val,y_val)
end_time = time.time()
print(f"Total time to generate submatrices: {end_time - start_time} seconds")


np.savez_compressed(f'./output/maceio/maceio_chrono',
                        x_train=x_t_sub,y_train=y_t_sub,x_val=x_v_sub,y_val=y_v_sub,i_train=index_sub_t,i_val=index_sub_v)



3. Generate random submatrices...
Sample 7: couldn't find submatrix! Count=3
Sample 14: couldn't find submatrix! Count=3
Sample 23: couldn't find submatrix! Count=1
Sample 148: couldn't find submatrix! Count=3
Sample 186: couldn't find submatrix! Count=4
Sample 231: couldn't find submatrix! Count=3
Sample 277: couldn't find submatrix! Count=1
Sample 308: couldn't find submatrix! Count=4
Sample 349: couldn't find submatrix! Count=3
Sample 379: couldn't find submatrix! Count=2
Sample 425: couldn't find submatrix! Count=4
Sample 1235: couldn't find submatrix! Count=4
Sample 1239: couldn't find submatrix! Count=2
Sample 1486: couldn't find submatrix! Count=4
Sample 1576: couldn't find submatrix! Count=3
Sample 1693: couldn't find submatrix! Count=4
Sample 2461: couldn't find submatrix! Count=3
Sample 2498: couldn't find submatrix! Count=3
Sample 2911: couldn't find submatrix! Count=2
Sample 2970: couldn't find submatrix! Count=4
Sample 2986: couldn't find submatrix! Count=3
Sample 3051: c

In [5]:

test = df_cri.loc[df_cri.index.tolist()[0]].values.tolist()
test.reverse()
#display(test)

test2 = [1, 1, 2, 3, 4, 6, 7, 8, 9]
display( test2)
test2.reverse()
display(np.expand_dims(np.array(test2).reshape(3,3,1).T,  axis=-1))

# single_image = np.random.rand(4, 4)
# display(single_image)

# # Reshape to (1, 39, 39, 1)
# cnn_input = single_image.reshape(1, 4, 4, 1)
# display(cnn_input)

[1, 1, 2, 3, 4, 6, 7, 8, 9]

array([[[[9],
         [6],
         [2]],

        [[8],
         [4],
         [1]],

        [[7],
         [3],
         [1]]]])